# Nettoyage et Normalisation des Données de Nœuds

Notebook pour nettoyer, normaliser et enrichir les données de formes de nœuds extraites du fichier d'événements.

**Objectifs** :
1. Filtrer les dates et valeurs temporelles mal normalisées
2. Normaliser accents, espaces et caractères spéciaux

In [ ]:
import json
import re
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

DATA_PATH = "data/export.events.json"

In [36]:
def safe_first_label(labels):
    if isinstance(labels, list) and len(labels) > 0:
        return labels[0]
    return None

def simplify_taxonomy(value, level=None):
    """
    Exemple :
    Thing/Abstract/Event/Transfer/TransferOfUnbiasedInformation

    level=None -> TransferOfUnbiasedInformation
    level=1    -> Thing
    level=2    -> Thing/Abstract
    """
    if not isinstance(value, str):
        return None
    parts = value.split("/")
    if level is None:
        return parts[-1]
    return "/".join(parts[:level])

def normalize_spacing(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

def lowercase_text(text):
    if not isinstance(text, str):
        return ""
    return normalize_spacing(text).lower()

In [37]:
def is_date_like(text):
    """
    Détecte des formes de type date / heure / timestamp.
    Détection volontairement large pour éliminer le bruit temporel.
    """
    if not isinstance(text, str):
        return False

    text = text.strip().lower()

    patterns = [
        r"^\d{4}-\d{2}-\d{2}",                          # 2026-02-10 ou 2026-02-10T...
        r"^\d{1,2}:\d{2}(:\d{2})?$",                   # 06:00 ou 06:00:00
        r"^\d{4}/\d{2}/\d{2}$",                        # 2026/02/10
        r"^\d{1,2}/\d{1,2}/\d{4}$",                    # 10/02/2026
        r"^\d{1,2}-\d{1,2}-\d{4}$",                    # 10-02-2026
        r"^\d{8}$",                                    # 20260210
        r"^\d{4}-\d{2}-\d{2}t\d{2}:\d{2}",             # 2026-02-10t13:42
        r"^0000-01-01t00:00",                          # bruit déjà observé
        r"^\d{1,2}(er)?\s?[a-zéû]+\b",                 # 1er janvier, 9fevrier
        r"^(lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche)\b",  # lundi...
    ]

    return any(re.search(pattern, text, re.IGNORECASE) for pattern in patterns)

In [38]:
def remove_accents(text):
    if not isinstance(text, str):
        return text
    text_nfd = unicodedata.normalize("NFD", text)
    return "".join(c for c in text_nfd if unicodedata.category(c) != "Mn")

def normalize_special_chars(text):
    """
    Normalisation légère.
    On conserve le plus possible l'information lexicale.
    """
    if not isinstance(text, str):
        return text

    # Uniformiser divers tirets
    text = re.sub(r"[\u2010-\u2015\u2212]", "-", text)

    # Apostrophes typographiques -> apostrophe simple
    text = text.replace("’", "'").replace("`", "'")

    # Réduction des espaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_form(text):
    """
    Pipeline simple de nettoyage lexical.
    """
    if not isinstance(text, str):
        return ""

    text = normalize_spacing(text)
    text = text.lower()
    text = remove_accents(text)
    text = normalize_special_chars(text)

    return text

In [39]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
date_count = 0
missing_form_count = 0

for ev in data:
    event_id = ev.get("_id", {}).get("$oid") if isinstance(ev.get("_id"), dict) else ev.get("_id")
    event_type = ev.get("type")
    result_analyse_id = ev.get("resultAnalyseId")
    task_id = ev.get("taskId")

    nodes = ev.get("nodes", [])
    if not isinstance(nodes, list):
        continue

    for node in nodes:
        if not isinstance(node, dict):
            continue

        form_original = node.get("form")
        if not isinstance(form_original, str) or not form_original.strip():
            missing_form_count += 1
            continue

        form_surface = normalize_spacing(form_original)
        node_label = safe_first_label(node.get("labels", []))
        node_label_last = simplify_taxonomy(node_label, None)

        if is_date_like(form_surface):
            date_count += 1
            continue

        rows.append({
            "event_id": event_id,
            "event_type": event_type,
            "event_type_last": simplify_taxonomy(event_type, None),
            "resultAnalyseId": result_analyse_id,
            "taskId": task_id,
            "node_id": node.get("_id"),
            "node_label": node_label,
            "node_label_last": node_label_last,
            "form_original": form_original,
            "form_surface": form_surface,
        })

df_forms = pd.DataFrame(rows)

print(f"Nombre total d'occurrences conservées : {len(df_forms)}")
print(f"Dates / formes temporelles filtrées : {date_count}")
print(f"Formes absentes ou vides : {missing_form_count}")
df_forms.head()

Nombre total d'occurrences conservées : 30334
Dates / formes temporelles filtrées : 5368
Formes absentes ou vides : 0


,event_id,event_type,event_type_last,resultAnalyseId,taskId,node_id,node_label,node_label_last,form_original,form_surface
0,698b35e78f03efde04a87998,Thing/Abstract/Event/Win,Win,698b35e042a8083a290c11c7,1a076139-ea5e-40e2-8d82-01a1f28b8181,474738074,Thing/Abstract/Event/Win,Win,remporter,remporter
1,698b35e78f03efde04a87998,Thing/Abstract/Event/Win,Win,698b35e042a8083a290c11c7,1a076139-ea5e-40e2-8d82-01a1f28b8181,2080660730,Thing/Concrete/Inanimate,Inanimate,fauteuil,fauteuil
2,698b35e98f03efde04a8799a,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,698b35e042a8083a290c11c5,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,1667751063,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,publiés,publiés
3,698b35e98f03efde04a8799a,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,698b35e042a8083a290c11c5,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,1664342503,Thing/Concrete/Inanimate/Product/Communication...,CommunicationMedium,e-mails,e-mails
4,698b35e98f03efde04a8799b,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,698b35e042a8083a290c11c5,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,562551427,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,témoignent,témoignent


In [40]:
df_forms["form_norm"] = df_forms["form_surface"].apply(lowercase_text)
df_forms["form_no_accent"] = df_forms["form_norm"].apply(remove_accents)
df_forms["form_cleaned"] = df_forms["form_no_accent"].apply(normalize_special_chars)

print("Colonnes de normalisation créées")
df_forms[["form_original", "form_surface", "form_norm", "form_cleaned"]].head(10)

Colonnes de normalisation créées


,form_original,form_surface,form_norm,form_cleaned
0,remporter,remporter,remporter,remporter
1,fauteuil,fauteuil,fauteuil,fauteuil
2,publiés,publiés,publiés,publies
3,e-mails,e-mails,e-mails,e-mails
4,témoignent,témoignent,témoignent,temoignent
5,publiés,publiés,publiés,publies
6,servi,servi,servi,servi
7,montages,montages,montages,montages
8,et,et,et,et
9,2011 et 2019,2011 et 2019,2011 et 2019,2011 et 2019


In [41]:
changed_accents = df_forms[df_forms["form_norm"] != df_forms["form_no_accent"]][
    ["form_original", "form_norm", "form_no_accent"]
].drop_duplicates().head(20)

changed_special = df_forms[df_forms["form_no_accent"] != df_forms["form_cleaned"]][
    ["form_original", "form_no_accent", "form_cleaned"]
].drop_duplicates().head(20)

print("Exemples de suppression d'accents :")
display(changed_accents)

print("\nExemples de normalisation de caractères spéciaux :")
display(changed_special)

Exemples de suppression d'accents :


,form_original,form_norm,form_no_accent
2,publiés,publiés,publies
4,témoignent,témoignent,temoignent
13,conférence de presse,conférence de presse,conference de presse
16,témoins,témoins,temoins
19,tué,tué,tue
31,élargit,élargit,elargit
34,contrôle,contrôle,controle
40,blessées,blessées,blessees
45,étaient,étaient,etaient
47,région,région,region



Exemples de normalisation de caractères spéciaux :


,form_original,form_no_accent,form_cleaned


In [42]:
def singularize_simple(word):
    """
    Règles prudentes et limitées.
    On ne cherche pas à faire une vraie lemmatisation.
    """
    if not isinstance(word, str):
        return word

    w = word.strip()
    if len(w) <= 3:
        return w

    # ies -> y
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"

    # formes en es
    if w.endswith("es") and len(w) > 4:
        return w[:-2]

    # formes en s
    if w.endswith("s") and len(w) > 3:
        return w[:-1]

    return w

all_clean_forms = set(df_forms["form_cleaned"].dropna().unique())

def canonical_form(form):
    """
    Ne fusionne que si la variante existe réellement dans les données.
    """
    if not isinstance(form, str):
        return form

    candidate = singularize_simple(form)

    if candidate != form and candidate in all_clean_forms:
        return candidate

    return form

df_forms["form_canonical"] = df_forms["form_cleaned"].apply(canonical_form)

df_forms[["form_cleaned", "form_canonical"]].drop_duplicates().head(20)

,form_cleaned,form_canonical
0,remporter,remporter
1,fauteuil,fauteuil
2,publies,publies
3,e-mails,e-mails
4,temoignent,temoignent
6,servi,servi
7,montages,montages
8,et,et
9,2011 et 2019,2011 et 2019
11,tenu,tenu


In [43]:
variant_table = (
    df_forms[["form_cleaned", "form_canonical"]]
    .drop_duplicates()
    .groupby("form_canonical")
    .agg(
        n_variants=("form_cleaned", "nunique"),
        variants=("form_cleaned", lambda x: sorted(list(x))[:20])
    )
    .reset_index()
)

variant_table = variant_table[variant_table["n_variants"] > 1].sort_values(
    "n_variants", ascending=False
)

print(f"Nombre de formes canoniques avec au moins 2 variantes : {len(variant_table)}")
variant_table.head(20)

Nombre de formes canoniques avec au moins 2 variantes : 445


,form_canonical,n_variants,variants
439,allemand,3,"[allemand, allemandes, allemands]"
485,americain,3,"[americain, americaines, americains]"
154,abattu,3,"[abattu, abattues, abattus]"
6285,rapport,3,"[rapport, rapportes, rapports]"
7422,suspect,3,"[suspect, suspectes, suspects]"
7426,suspendu,3,"[suspendu, suspendues, suspendus]"
7526,tenu,3,"[tenu, tenues, tenus]"
6507,rejet,3,"[rejet, rejetes, rejets]"
6280,rappel,3,"[rappel, rappeles, rappels]"
5783,port,3,"[port, portes, ports]"


In [44]:
def is_entity_like(form_surface):
    """
    Détection simple d'entités de surface à partir de la casse originale.
    """
    if not isinstance(form_surface, str):
        return False

    tokens = [t for t in form_surface.split() if t]
    if not tokens:
        return False

    # au moins un token avec majuscule initiale
    return any(tok[:1].isupper() for tok in tokens)

def is_verb_like(form_cleaned):
    """
    Heuristique simple pour repérer des formes verbales.
    Ce n'est pas une POS complète.
    """
    if not isinstance(form_cleaned, str):
        return False

    common_verbs = {
        "est", "sont", "etre", "avoir", "faire", "aller", "venir", "pouvoir",
        "devoir", "vouloir", "faut", "fait", "dit", "peut", "doit",
        "is", "are", "was", "were", "be", "have", "has", "do", "does",
        "go", "goes", "make", "makes", "come", "comes"
    }

    if form_cleaned in common_verbs:
        return True

    verb_suffixes = (
        "er", "ir", "re", "oir",
        "e", "es", "ent",
        "ait", "aient",
        "era", "eront",
        "ant",
        "é", "ée", "és", "ées",
        "i", "ie", "is", "it",
        "u", "ue", "us",
        "ing", "ed", "ate", "ified"
    )

    return form_cleaned.endswith(verb_suffixes)

def classify_form_type(form_surface, form_cleaned):
    if not isinstance(form_cleaned, str) or not form_cleaned.strip():
        return "OTHER"

    if is_date_like(form_cleaned):
        return "TIME_LIKE"

    if is_entity_like(form_surface):
        return "ENTITY_LIKE"

    if is_verb_like(form_cleaned):
        return "VERB_LIKE"

    return "OTHER"

df_forms["form_type"] = df_forms.apply(
    lambda row: classify_form_type(row["form_surface"], row["form_cleaned"]),
    axis=1
)

df_forms["form_type"].value_counts(dropna=False)

form_type
VERB_LIKE      17823
OTHER           8667
ENTITY_LIKE     3842
TIME_LIKE          2
Name: count, dtype: int64

In [45]:
print("Exemples ENTITY_LIKE :")
display(
    df_forms[df_forms["form_type"] == "ENTITY_LIKE"][
        ["form_surface", "form_cleaned", "node_label_last"]
    ].drop_duplicates().head(20)
)

print("\nExemples VERB_LIKE :")
display(
    df_forms[df_forms["form_type"] == "VERB_LIKE"][
        ["form_surface", "form_cleaned", "node_label_last"]
    ].drop_duplicates().head(20)
)

print("\nExemples OTHER :")
display(
    df_forms[df_forms["form_type"] == "OTHER"][
        ["form_surface", "form_cleaned", "node_label_last"]
    ].drop_duplicates().head(20)
)

Exemples ENTITY_LIKE :


,form_surface,form_cleaned,node_label_last
44,Par ailleurs,par ailleurs,Addition
52,Jeux olympiques,jeux olympiques,Organization
62,Maroc,maroc,Country
70,Cuba,cuba,Place
143,Constitution,constitution,CommunicationMedium
149,Receiver,receiver,Thing
156,Atlantique,atlantique,Place
167,Marseille,marseille,Place
188,Dunkerque,dunkerque,Place
189,Emmanuel Macron,emmanuel macron,Civilian



Exemples VERB_LIKE :


,form_surface,form_cleaned,node_label_last
0,remporter,remporter,Win
2,publiés,publies,TransferOfUnbiasedInformation
4,témoignent,temoignent,TransferOfUnbiasedInformation
6,servi,servi,Thing
7,montages,montages,Build
11,tenu,tenu,Hold
13,conférence de presse,conference de presse,Concrete
14,diffuser,diffuser,Thing
17,reconnu,reconnu,Confess
18,homme,homme,Human



Exemples OTHER :


,form_surface,form_cleaned,node_label_last
1,fauteuil,fauteuil,Inanimate
3,e-mails,e-mails,CommunicationMedium
8,et,et,Addition
9,2011 et 2019,2011 et 2019,Time
12,procureur,procureur,Civilian
15,appel,appel,Communication
16,témoins,temoins,Human
20,sa,sa,Own
23,faits,faits,Abstract
24,et,et,Thing


In [46]:
df_forms_unique = (
    df_forms[
        [
            "form_original",
            "form_surface",
            "form_norm",
            "form_no_accent",
            "form_cleaned",
            "form_canonical",
            "node_label",
            "node_label_last",
            "form_type"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print(f"Nombre de formes uniques après nettoyage : {len(df_forms_unique)}")
df_forms_unique.head()

Nombre de formes uniques après nettoyage : 11764


,form_original,form_surface,form_norm,form_no_accent,form_cleaned,form_canonical,node_label,node_label_last,form_type
0,remporter,remporter,remporter,remporter,remporter,remporter,Thing/Abstract/Event/Win,Win,VERB_LIKE
1,fauteuil,fauteuil,fauteuil,fauteuil,fauteuil,fauteuil,Thing/Concrete/Inanimate,Inanimate,OTHER
2,publiés,publiés,publiés,publies,publies,publies,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,VERB_LIKE
3,e-mails,e-mails,e-mails,e-mails,e-mails,e-mails,Thing/Concrete/Inanimate/Product/Communication...,CommunicationMedium,OTHER
4,témoignent,témoignent,témoignent,temoignent,temoignent,temoignent,Thing/Abstract/Event/Transfer/TransferOfUnbias...,TransferOfUnbiasedInformation,VERB_LIKE


In [52]:
# Créer un mapping form_original -> form_canonical pour appliquer au fichier JSON
form_mapping = dict(zip(df_forms['form_original'], df_forms['form_canonical']))

print(f"Mapping créé: {len(form_mapping)} formes mappées")
print(f"Exemples: {list(form_mapping.items())[:5]}")

# Charger le JSON original
with open(DATA_PATH, "r", encoding="utf-8") as f:
    original_data = json.load(f)

# Appliquer les transformations aux nœuds
cleaned_data = []
cleaned_count = 0
unchanged_count = 0

for event in original_data:
    event_cleaned = event.copy()
    if "nodes" in event_cleaned:
        new_nodes = []
        for node in event_cleaned["nodes"]:
            if isinstance(node, dict) and "form" in node:
                original_form = node["form"]
                # Chercher la forme nettoyée
                if original_form in form_mapping:
                    node_cleaned = node.copy()
                    node_cleaned["form"] = form_mapping[original_form]
                    new_nodes.append(node_cleaned)
                    cleaned_count += 1
                else:
                    # Forme non trouvée dans le mapping (ex: date filtrée), ignorer
                    pass
            else:
                new_nodes.append(node)
                unchanged_count += 1
        event_cleaned["nodes"] = new_nodes
    
    cleaned_data.append(event_cleaned)

# Sauvegarder le fichier nettoyé
output_clean_json = "data/export.events.clean.json"
with open(output_clean_json, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print(f"Fichier JSON nettoyé généré: {output_clean_json}")
print(f"  Formes nettoyées: {cleaned_count}")
print(f"  Formes conservées: {unchanged_count}")
print(f"  Total événements: {len(cleaned_data)}")
print(f"{'='*60}")

Mapping créé: 9081 formes mappées
Exemples: [('remporter', 'remporter'), ('fauteuil', 'fauteuil'), ('publiés', 'publies'), ('e-mails', 'e-mails'), ('témoignent', 'temoignent')]

Fichier JSON nettoyé généré: data/export.events.clean.json
  Formes nettoyées: 30334
  Formes conservées: 0
  Total événements: 11948
